In [ ]:
import pandas as pd 
import numpy as np
import pickle as pk 
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pd.options.display.max_columns = 100
pd.options.display.max_rows = 60

# Preface: Evaluating fairness tradeoffs in fairness metrics 
The objective of "fairness" in ML often involves tradeoffs. In fact, there are several well-known “impossibility theorems” (e.g., https://arxiv.org/abs/1609.07236) showing that certain fairness criteria (such as calibration and equalized odds) cannot all be satisfied at the same time when groups have different base rates.  

Similarly, there is often a tradeoff between training a "fair" model and training a highly accurate model. For example, enforcing fairness constraints may require the model to ignore predictive signals that differ across groups, which can reduce overall performance.


In this notebook, you will:
1. Examine the tradeoffs that arise between fairness and performance
2. Examine the tradeoffs between fairness metrics themselves 
3. Apply simple fairness interventions to make the model more fair with respect to a chosen metric  
4. Evaluate how these interventions affect overall model performance  


In [ ]:
patient_agg = pd.read_parquet('../../data/clean_dataset.parquet')
feature_sets = pk.load(open('../../data/feature_names.pkl','rb'))

# This distinction of numerical (continuous), binary, and categorical columns may be useful for you 
num_cols = feature_sets['labs'].tolist() + feature_sets['vitals'].tolist() + ['age', 'admissionheight', 'admissionweight']
bin_cols = feature_sets['icd10_before24h'].tolist() + feature_sets['admissiondx'].tolist()
cat_cols = ['hospital_region', 'ethnicity', 'gender', 'hospital_numbedscategory', 'hospitaldischargeyear', 'hospitalid']

# For privacy I think, the dataset sets the age of everyone over 89 as the string '>89'. We set back to a continuous number 
if 'age' in patient_agg.columns: 
    patient_agg.loc[patient_agg.age == '> 89', 'age'] = 90
    patient_agg['age'] = patient_agg['age'].astype(float)

Y_label = 'mortality_at48h'
X_covariate_names = num_cols + bin_cols + cat_cols

X = patient_agg[X_covariate_names].copy()
Y = patient_agg[Y_label].copy()

# TODO: apply the same feature selection steps as in week2 
X

In [ ]:
# TODO: Learn a model to predict Y from X 

# Q1. Tradeoffs between fairness metrics 
Next, we will examine how fairness metrics are not always consistent. That is, a model may be fair with respect to one metric but unfair with respect to another.  
We will explore how these fairness metrics change across different binary thresholds, where predictions are defined as:

$$
\hat{Y} = \mathbf{1}(r > t)
$$

In [ ]:
# TODO: As you did in week2, evaluate the performance of your model using different fairness metrics.
# Vary the threshold for Yhat 
# For example, pseudocode would be : 

# yhat_proba = model.predict_proba(X)[:, 1]
# for threshold in np.linspace(0, 1, num=5): 
#     yhat = (yhat_proba >= threshold).astype(int)
#     overall_accuracy = ... (we will use in Q2)

#     for a in A: 
#         Compute EO(a, yhat, y), DP(a, yhat, y)
        
# Compare the performance of your model across different thresholds and different fairness metrics.

In [ ]:
# TODO: Discuss the tradeoffs you observe between different fairness metrics. 

# Q2. Tradeoffs between fairness and accuracy 
Next, we'll examine how a model that might be fair with respect to a certain fairness metric may be overall less accurate. 

In [ ]:
# TODO: Plot overall_accuracy vs some fairness metric (e.g., demographic parity) and discuss your observations 
# eg. plt.scatter(fairness_gap, accuracy)

# Q3. Learning a more fair model: reweighting 
Think critically about what subgroups you think 'deserves' better performance from the model. 

One way to learn a more "fair" model for that subgroup is to upweight it during training. 
For example, assign a larger training weight to examples from the subgroup with the worst performance.
Hint: Many sklearn models accept a `sample_weight` argument during `.fit()`.

Then, relearn the model using these new sample weights and assess performance again. 

In [ ]:
# TODO: Relearn the model with different sample weights, and report the same metrics as above in Q1 and Q2.
# Did you observe any changes in overall accuracy, or how any fairness metric changed? 
# Which groups were harmed? Which groups benefited? 

In [ ]:
# TODO: Reflect. Do you think you would deploy the original model for mortality prediction, or this new model? Why? 

## Q4. Brainstorming a fairness goal

Let's dive a bit deeper into what a reasonable fairness objective is for this model. 
Examples:
- Reduce the gap in false negative rate between groups
- Improve worst-group AUROC
- Reduce the gap in recall between groups
- Improve performance for the subgroup with the lowest accuracy
And for what group? Thinking critically about what you are trying to achieve with your model is an important part of this project. 


In [ ]:
# TODO: With your group, brainstorm specific ideas of what "fair" means to you for mortality prediction. 
# Is this captured by existing metrics? 